In [ ]:
from iFinDPy import *
import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL
from datetime import date
import pandas as pd
import numpy as np
from QGI.neoapi import Neo4j

In [ ]:

def get_logger():
    logger=logging.getLogger('logger')
    logger.setLevel(logging.INFO)
    fh=logging.FileHandler("E:/wangzhilin/QuantumGalaxy/logs/aura_api_log.log",'a',encoding='utf-8')
    fh.setLevel(logging.INFO)
    ch=logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter=logging.Formatter(fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    fh.setFormatter(formatter)
    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger
logger=get_logger()

In [ ]:
def update_from_name():
    neo_uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
    neo_user = "QG_Editor"
    neo_password = "editor"
    neo=Neo4j(neo_uri,neo_user,neo_password)
    sql_host='localhost'
    sql_user='Local_Editor'
    sql_password='QuantumGalaxy'
    database='qgdbs'
    mysql=MYSQL(sql_host,sql_user,sql_password,database)
    code=neo.fetch_company_with_code()
    today=date.today()-timedelta(1)
    today=today.__format__('%y-%m-%d')
    codestr=str(code)
    codestr1='('+codestr[1:-1]+')'
    sql="select code,close_price from ticker_data where date='%s' and code in %s"%(today,codestr1)
    r=mysql.read_query(sql)
    neo.update_company_value(data=r)

    sql="select code, stdchg3m from processed_data where date='%s' and code in %s"%(today,codestr1)
    result=mysql.read_query(sql)
    neo.update_company_stdcgh3m(data=result)

    neo.add_indicator_to_company()
    neo.close()
    mysql.close()

In [11]:
def update_from_code(neo):
    #neo_uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
    #neo_user = "QG_Editor"
    #neo_password = "editor"
    #neo=Neo4j(neo_uri,neo_user,neo_password)
    #uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
    #user = "neo4j"
    #password = "QuantumGalaxy"
    #neo=Neo4j(uri,user,password)
    sql_host='localhost'
    sql_user='Local_Editor'
    sql_password='QuantumGalaxy'
    database='qgdbs'
    mysql=MYSQL(sql_host,sql_user,sql_password,database)
    code=neo.fetch_company_and_indicator_with_code()
    strip_code=[]
    for c in code:
        strip_code.append(c.replace('\t','').replace('\n',''))
    today=date.today()-timedelta(0)
    today=today.__format__('%y-%m-%d')
    codestr=str(strip_code)
    codestr1='('+codestr[1:-1]+')'
    df=pd.DataFrame(index=code)
    sql='update ticker_info set status=1 where code in %s'%codestr1
    result=mysql.write_query(sql)
    sql="select code,close_price from ticker_data where date='%s' and code in %s and not close_price is null  "%(today,codestr1)
    r=mysql.read_query(sql)
    df1=pd.DataFrame(data=r,columns=['code','close']).set_index('code')
    df=pd.merge(df,df1,'outer',left_index=True,right_index=True)
    print(len(r))
    #neo.update_node_value(data=r)
    sql="select code,date from ticker_data where date='%s' and code in %s   "%(today,codestr1)
    r=mysql.read_query(sql)
    df1=pd.DataFrame(data=r,columns=['code','date']).set_index('code')
    df=pd.merge(df,df1,'outer',left_index=True,right_index=True)
    #neo.update_node_data_date(data=r)
    sql="select code, market_value2 from ticker_data where date='%s' and code in %s and not market_value2 is null"%(today,codestr1)
    r=mysql.read_query(sql)
    df1=pd.DataFrame(data=r,columns=['code','market_value']).set_index('code')
    df=pd.merge(df,df1,'outer',left_index=True,right_index=True)
    #neo.update_node_market_value(data=r)
    sql="select code, stdchg3m from processed_data where date='%s' and code in %s  and not stdchg3m is null"%(today,codestr1)
    r=mysql.read_query(sql)
    df1=pd.DataFrame(data=r,columns=['code','stdchg3m']).set_index('code')
    df=pd.merge(df,df1,'outer',left_index=True,right_index=True)
    #neo.update_node_stdchg3m(data=result)
    sql="select code, stdchg1m from processed_data where date='%s' and code in %s and not stdchg1m is null"%(today,codestr1)
    r=mysql.read_query(sql)
    df1=pd.DataFrame(data=r,columns=['code','stdchg1m']).set_index('code')
    df=pd.merge(df,df1,'outer',left_index=True,right_index=True)
    #neo.update_node_stdchg1m(data=result)
    sql="select code, change_rate_1d from processed_data where date='%s' and code in %s and not change_rate_1d is null"%(today,codestr1)
    r=mysql.read_query(sql)
    df1=pd.DataFrame(data=r,columns=['code','stdchg1d']).set_index('code')
    df=pd.merge(df,df1,'outer',left_index=True,right_index=True)
    #neo.update_node_chg1d(data=result)
    sql='''SELECT t1.code,t1.stdchg1m - t2.stdchg1m AS diff \
    FROM processed_data t1 \
    JOIN processed_data t2 ON t1.code = t2.code \
    WHERE t1.date = '%s' AND t2.date = ( \
        SELECT MAX(date) \
        FROM processed_data \
        WHERE date <= DATE_SUB('%s', INTERVAL 7 DAY) AND code = t1.code \
    ) \
    AND t1.code  in %s'''%(today,today,codestr1)
    r=mysql.read_query(sql)
    df1=pd.DataFrame(data=r,columns=['code','chg1m_1w_diff']).set_index('code')
    df=pd.merge(df,df1,'outer',left_index=True,right_index=True)
    #neo.add_indicator_to_company()
    neo.close()
    mysql.close()
    df=df.dropna(how='all').round(2)
    cypher_list=[]
    for i,r in df.head().iterrows():
        
        cypher=f'''match (n:Company|Indicator {{code:'{i}'}}) set '''
        fields=''
        for k,v in r.items():
            if k=='date':
                fields+='n.'+k+'='+"'"+(str(v) if v else 'null')+"'"+','
            else:
                fields+='n.'+k+'='+(str(v) if v else 'null')+','
        cypher+=fields[:-1]
        print(cypher)
        cypher_list.append(cypher)
    r=neo.multi_cypher(cypher_list)  
    return df

In [12]:
df=update_from_code(neo)


04-27 21:32 logger       INFO     using username Local_Editor
04-27 21:32 logger       INFO     accessed to database <qgdbs>
04-27 21:32 logger       INFO     got a cursor
04-27 21:32 logger       INFO     Executed sql "update ticker_info set status=1 where code in ('002739.SZ', '600977.SH', '688007.SH', '002905.SZ', '300133.SZ', '002343.SZ', '300413.SZ', '300264.SZ', '000892.SZ', '002712.SZ', '600088.SH', '300027.SZ', '002027.SZ', '002292.SZ', '002917.SZ', '1060.HK', '300182.SZ', '600136.SH', '603000.SH', '600959.SH', '300624.SZ', '300291.SZ', '600052.SH', '300459.SZ', '300251.SZ', '人工', '000681.SZ', '605168.SH', '600228.SH', '600715.SH', '601595.SH', '601599.SH', '300833.SZ', '000802.SZ', '0009.HK', '002445.SZ', '002624.SZ', '0391.HK', '0419.HK', '0491.HK', '0571.HK', '0745.HK', '0764.HK', '1046.HK', '1132.HK', '1159.HK', '1326.HK', '1740.HK', '1859.HK', '1970.HK', '1981.HK', '2125.HK', '8271.HK', '8280.HK', '603103.SH', '6698.HK', '8078.HK', '8082.HK', '8112.HK', '8130.HK', '8172.HK

2562


04-27 21:32 logger       INFO     Accepted read sql query "select code, stdchg3m from processed_data where date='23-04-27' and code in ('002739.SZ', '600977.SH', '688007.SH', '002905.SZ', '300133.SZ', '002343.SZ', '300413.SZ', '300264.SZ', '000892.SZ', '002712.SZ', '600088.SH', '300027.SZ', '002027.SZ', '002292.SZ', '002917.SZ', '1060.HK', '300182.SZ', '600136.SH', '603000.SH', '600959.SH', '300624.SZ', '300291.SZ', '600052.SH', '300459.SZ', '300251.SZ', '人工', '000681.SZ', '605168.SH', '600228.SH', '600715.SH', '601595.SH', '601599.SH', '300833.SZ', '000802.SZ', '0009.HK', '002445.SZ', '002624.SZ', '0391.HK', '0419.HK', '0491.HK', '0571.HK', '0745.HK', '0764.HK', '1046.HK', '1132.HK', '1159.HK', '1326.HK', '1740.HK', '1859.HK', '1970.HK', '1981.HK', '2125.HK', '8271.HK', '8280.HK', '603103.SH', '6698.HK', '8078.HK', '8082.HK', '8112.HK', '8130.HK', '8172.HK', '8220.HK', '300528.SZ', '300426.SZ', '002758.SZ', '603029.SH', '000998.SZ', '001338.SZ', '003042.SZ', '002749.SZ', '600354.SH', 

match (n:Company|Indicator {code:'000001.SZ'}) set n.close=12.26,n.date='2023-04-27',n.market_value=null,n.stdchg3m=-1.51,n.stdchg1m=-0.38,n.stdchg1d=0.01,n.chg1m_1w_diff=-0.38
match (n:Company|Indicator {code:'000002.SZ'}) set n.close=14.92,n.date='2023-04-27',n.market_value=null,n.stdchg3m=-1.67,n.stdchg1m=-0.37,n.stdchg1d=0.01,n.chg1m_1w_diff=0.03
match (n:Company|Indicator {code:'000004.SZ'}) set n.close=10.51,n.date='2023-04-27',n.market_value=null,n.stdchg3m=0.26,n.stdchg1m=0.47,n.stdchg1d=0.03,n.chg1m_1w_diff=-0.8
match (n:Company|Indicator {code:'000006.SZ'}) set n.close=4.70,n.date='2023-04-27',n.market_value=null,n.stdchg3m=-0.85,n.stdchg1m=-0.96,n.stdchg1d=null,n.chg1m_1w_diff=-0.15
match (n:Company|Indicator {code:'000008.SZ'}) set n.close=2.28,n.date='2023-04-27',n.market_value=null,n.stdchg3m=-0.49,n.stdchg1m=-0.72,n.stdchg1d=null,n.chg1m_1w_diff=-0.17


04-27 21:32 logger       INFO     write and commit 5  records


In [ ]:

df['date'].dt


In [ ]:
df.columns

In [ ]:

cypher_list=[]
for i,r in df.head().iterrows():
    
    cypher=f'''match (n:Company|Indicator {{code:'{i}'}}) set '''
    fields=''
    for k,v in r.items():
        if k=='date':
            fields+='n.'+k+'='+"'"+(str(v) if v else 'null')+"'"+','
        else:
            fields+='n.'+k+'='+(str(v) if v else 'null')+','
    cypher+=fields[:-1]
    print(cypher)
    cypher_list.append(cypher)
    

In [ ]:
r=neo.multi_cypher(cypher_list)

In [ ]:
r

In [ ]:
fields=''
for k,v in r.items():
    fields+='n.'+k+'='+(str(v) if v else 'null')+','
fields=fields[:-1]
fields

In [ ]:
sql_host='localhost'
sql_user='Local_Editor'
sql_password='QuantumGalaxy'
database='qgdbs'
mysql=MYSQL(sql_host,sql_user,sql_password,database)

In [ ]:
uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
user = "neo4j"
password = "QuantumGalaxy"
neo=Neo4j(uri,user,password)

In [ ]:
neo.__doc__

In [ ]:
neo.muti_cypher()

In [ ]:
code=neo.fetch_company_and_indicator_with_code()
strip_code=[]
for c in code:
    strip_code.append(c.replace('\t','').replace('\n',''))
today=date.today()-timedelta(0)
today=today.__format__('%y-%m-%d')
codestr=str(strip_code)
codestr1='('+codestr[1:-1]+')'

In [ ]:
sql='''SELECT t1.code,t1.stdchg1m - t2.stdchg1m AS diff \
FROM processed_data t1 \
JOIN processed_data t2 ON t1.code = t2.code \
WHERE t1.date = '%s' AND t2.date = ( \
    SELECT MAX(date) \
    FROM processed_data \
    WHERE date <= DATE_SUB('%s', INTERVAL 7 DAY) AND code = t1.code \
) \
AND t1.code  in %s'''%(today,today,codestr1)
result=mysql.read_query(sql)

In [ ]:
import pandas as pd
df=pd.DataFrame(index=code)


In [ ]:
df1=pd.DataFrame(data=result,columns=['code','diff']).set_index('code')
r=pd.merge(df,df1,'outer',left_index=True,right_index=True)

In [ ]:
r=pd.merge(df,df1,'outer',left_index=True,right_index=True)
r[pd.isna(r['diff'])]

In [ ]:
df

In [ ]:



def update_from_code():
    neo_uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
    neo_user = "neo4j"
    neo_password = "QuantumGalaxy"
    neo=Neo4j(neo_uri,neo_user,neo_password)
    sql_host='localhost'
    sql_user='Local_Editor'
    sql_password='QuantumGalaxy'
    database='qgdbs'
    mysql=MYSQL(sql_host,sql_user,sql_password,database)
    code=neo.fetch_company_and_indicator_with_code()
    strip_code=[]
    for c in code:
        strip_code.append(c.replace('\t','').replace('\n',''))
    today=date.today()-timedelta(1)
    today=today.__format__('%y-%m-%d')
    codestr=str(strip_code)
    codestr1='('+codestr[1:-1]+')'
    sql='update ticker_info set status=1 where code in %s'%codestr1
    result=mysql.write_query(sql)
    sql="select code,close_price from ticker_data where date='%s' and code in %s and not close_price is null  "%(today,codestr1)
    r=mysql.read_query(sql)

    neo.update_node_value(data=r)
    sql="select code,date from ticker_data where date='%s' and code in %s   "%(today,codestr1)
    r=mysql.read_query(sql)
    neo.update_node_data_date(data=r)
    sql="select code, market_value2 from ticker_data where date='%s' and code in %s and not market_value2 is null"%(today,codestr1)
    r=mysql.read_query(sql)
    neo.update_node_market_value(data=r)
    sql="select code, stdchg3m from processed_data where date='%s' and code in %s  and not stdchg3m is null"%(today,codestr1)
    result=mysql.read_query(sql)
    neo.update_node_stdcgh3m(data=result)
    sql="select code, stdchg1m from processed_data where date='%s' and code in %s and not stdchg1m is null"%(today,codestr1)
    result=mysql.read_query(sql)
    neo.update_node_stdcgh1m(data=result)
    sql="select code, change_rate_1d from processed_data where date='%s' and code in %s and not change_rate_1d is null"%(today,codestr1)
    result=mysql.read_query(sql)
    neo.update_node_chg1d(data=result)
    #neo.add_indicator_to_company()
    neo.close()
    mysql.close()



In [ ]:
def update_wind_edb():
    neo_uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
    neo_user = "neo4j"
    neo_password = "QuantumGalaxy"
    neo=Neo4j(neo_uri,neo_user,neo_password)
    sql_host='localhost'
    sql_user='Local_Editor'
    sql_password='QuantumGalaxy'
    database='qgdbs'
    mysql=MYSQL(sql_host,sql_user,sql_password,database)
    codes=neo.read_query('MATCH (n:Indicator) WHERE n.code <> ""  RETURN  n.code as code')
    codes=[r['code'] for r in codes]
    for code in codes:
        update_1_edb(code,neo,mysql)
    neo.close()
    mysql.close()
    return 1


In [ ]:
update_wind_edb()

In [ ]:
def update_1_edb(code,neo,mysql):
    r=mysql.read_query('select code,date,value from wind_edb_data where code ="%s" and not value is null order by date desc limit 2'%code)
    if len(r)==2:
        chg=(r[0][2]-r[1][2])/r[1][2]
        chg=round(chg,2)
        data=list(r[0])
        data.append(chg)
        cypher='match (n:Indicator) where n.code="%s" set n.datadate="%s",n.value="%s",n.chgrate="%s"'%tuple(d for d in data)
        neo.simple_query(cypher)
    else:
        logger.info('"%s" edb indicator has no last 2 record in qgdbs'%code)

In [ ]:
r=mysql.read_query('select code,date,value from wind_edb_data where code ="%s" and not value is null order by date desc limit 2'%c)
r

In [ ]:
if len(r)==2:
    chg=(r[0][2]-r[1][2])/r[1][2]
    chg=round(chg,2)
    data=list(r[0])
    data.append(chg)
data

In [ ]:
cypher='match (n:Indicator) where n.code="%s" set n.datadate="%s",n.value="%s",n.chgrate="%s"'%tuple(d for d in data)
cypher

In [ ]:
neo.simple_query(cypher)

In [ ]:
neo_uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
neo_user = "neo4j"
neo_password = "QuantumGalaxy"
neo=Neo4j(neo_uri,neo_user,neo_password)

In [ ]:
update_from_code()

In [ ]:
update_from_code_1day()

In [ ]:
def job():
    update_from_code_1day()
    update_from_code()

In [ ]:
from base64 import b64encode

In [ ]:
job()

In [ ]:

from apscheduler.schedulers.background import BackgroundScheduler
job_defaults = { 'max_instances': 10 }
scheduler = BackgroundScheduler(job_defaults=job_defaults)

scheduler.start()
scheduler.add_job(update_from_code_1day, 'cron', day_of_week='mon-sun', hour=8, minute=28)
scheduler.add_job(update_from_code, 'cron', day_of_week='mon-sun', hour=16, minute=58)

In [ ]:
scheduler.print_jobs()

In [ ]:
scheduler.shutdown()

In [ ]:
valdata=[]
datedata=[]
std3mdata=[]
std1mdata=[]
for c in strip_code:
    sql='select close_price,date from ticker_data where code="%s" order by date desc limit 1'%c
    result=mysql.read_query(sql)
    if result:
        close,t=float(result[0][0]),result[0][1]
        val.append((c,close))
        #print(data0)

        datedata.append((c,t))
        #print(data0)
        
        break
neo.update_company_value(data0)
neo.update_company_data_date(data1)

#update_company_data_date(data=(c,))
    

In [ ]:
d='as','df']
a,b=d



b